# 07 · Gold dimensions
dimdate + 6 dims (module 07 §2). Full rebuild via `INSERT OVERWRITE`, so masks, tags and grants survive.

✅ Expect dimdate 6,574 · dimworker 2,693 · dimorganization 40 · dimjob 60 · dimgrade 12 · dimlocation 11 · dimlegalentity 5.

In [ ]:
from hcm_utils import load_config, latest_batch, rebuild

dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")
cfg = load_config()
s, g = f"{catalog}.silver", f"{catalog}.gold"
as_of = cfg["snapshot"]["end_date"]

for t in ["hr_all_organization_units", "per_locations_f", "per_legal_employers"]:
    latest_batch(f"{catalog}.bronze.{t}").createOrReplaceTempView(f"v_{t}")

results = {}

In [ ]:
# dimdate
results["dimdate"] = rebuild(f"{g}.dimdate", f"""
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS DateKey,
       d                                         AS Date,
       year(d)                                   AS Year,
       quarter(d)                                AS Quarter,
       month(d)                                  AS Month,
       date_format(d, 'MMMM')                    AS MonthName,
       date_format(d, 'yyyy-MM')                 AS YearMonth,
       dayofweek(d)                              AS DayOfWeek,
       d = last_day(d)                           AS IsMonthEnd
FROM (SELECT explode(sequence(DATE'{cfg["snapshot"]["dimdate_start"]}',
                              DATE'{cfg["snapshot"]["dimdate_end"]}', INTERVAL 1 DAY)) AS d)
""")

In [ ]:
# dimworker (current state)
results["dimworker"] = rebuild(f"{g}.dimworker", f"""
SELECT xxhash64(w.PersonId)                    AS WorkerKey,
       w.PersonId, w.PersonNumber, w.FullName, w.DateOfBirth,
       CASE WHEN w.DateOfBirth IS NULL THEN 'Unknown'
            WHEN months_between(DATE'{as_of}', w.DateOfBirth) / 12 < 25 THEN '<25'
            WHEN months_between(DATE'{as_of}', w.DateOfBirth) / 12 < 35 THEN '25-34'
            WHEN months_between(DATE'{as_of}', w.DateOfBirth) / 12 < 45 THEN '35-44'
            WHEN months_between(DATE'{as_of}', w.DateOfBirth) / 12 < 55 THEN '45-54'
            ELSE '55+' END                       AS AgeBand,
       CASE w.Gender WHEN 'M' THEN 'Male' WHEN 'F' THEN 'Female' ELSE 'Unknown' END AS Gender,
       w.PersonType,
       CASE w.PersonType WHEN 'EMP' THEN 'Employee' WHEN 'CWK' THEN 'Contingent' ELSE 'Unknown' END AS WorkerType,
       w.NationalIdentifier,
       p.DateStart                               AS HireDate,
       p.ActualTerminationDate                   AS TerminationDate,
       (p.ActualTerminationDate IS NULL OR p.ActualTerminationDate > DATE'{as_of}') AS IsActive
FROM {s}.hcmworker w
LEFT JOIN {s}.hcmperiodsofservice p ON p.PersonId = w.PersonId
WHERE w.IsCurrent
""")

In [ ]:
# dimorganization (flattened 2-level hierarchy)
results["dimorganization"] = rebuild(f"{g}.dimorganization", """
WITH o AS (
  SELECT try_cast(ORGANIZATION_ID AS INT) AS OrganizationId,
         trim(NAME) AS OrganizationName,
         trim(TYPE) AS OrganizationType,
         try_cast(try_cast(PARENT_ORGANIZATION_UNIT_ID AS DOUBLE) AS INT) AS ParentOrganizationId
  FROM v_hr_all_organization_units)
SELECT xxhash64(o.OrganizationId) AS OrganizationKey,
       o.OrganizationId, o.OrganizationName, o.OrganizationType, o.ParentOrganizationId,
       coalesce(p.OrganizationName, o.OrganizationName) AS BusinessUnitName
FROM o LEFT JOIN o p ON p.OrganizationId = o.ParentOrganizationId
""")

In [ ]:
# dimjob / dimgrade (current rows of SCD2 silver)
results["dimjob"] = rebuild(f"{g}.dimjob", f"""
SELECT xxhash64(JobId) AS JobKey, JobId, JobName, JobFamily
FROM {s}.hcmjob WHERE IsCurrent
""")
results["dimgrade"] = rebuild(f"{g}.dimgrade", f"""
SELECT xxhash64(GradeId) AS GradeKey, GradeId, GradeName, GradeLevel
FROM {s}.hcmgrade WHERE IsCurrent
""")

In [ ]:
# dimlocation (+ Unknown member)
results["dimlocation"] = rebuild(f"{g}.dimlocation", """
SELECT xxhash64(try_cast(LOCATION_ID AS INT)) AS LocationKey,
       try_cast(LOCATION_ID AS INT) AS LocationId,
       trim(LOCATION_CODE) AS LocationCode, trim(COUNTRY) AS Country,
       trim(REGION_1) AS Region, trim(TOWN_OR_CITY) AS City
FROM v_per_locations_f
UNION ALL
SELECT -1, -1, 'UNKNOWN_LOCATION', 'Unknown', 'Unknown', 'Unknown'
""")

# dimlegalentity (+ Unknown member)
results["dimlegalentity"] = rebuild(f"{g}.dimlegalentity", """
SELECT xxhash64(try_cast(LEGAL_ENTITY_ID AS INT)) AS LegalEntityKey,
       try_cast(LEGAL_ENTITY_ID AS INT) AS LegalEntityId,
       trim(NAME) AS LegalEntityName,
       trim(LEGISLATIVE_DATA_GROUP) AS LegislativeDataGroup,
       split(trim(LEGISLATIVE_DATA_GROUP), '_')[0] AS CountryCode
FROM v_per_legal_employers
UNION ALL
SELECT -1, -1, 'Unknown', 'Unknown', 'Unknown'
""")

print(results)